# MovieLens 100K Offline Benchmark

This notebook compares three recommenders on implicit feedback built from MovieLens 100K ratings (`rating >= 4`):

1. Item-item collaborative filtering (cosine similarity)
2. Matrix factorization (truncated SVD)
3. Deep learning pointwise ranker (embedding + MLP in PyTorch)

Evaluation uses a per-user temporal split (last 10% interactions held out) and reports **MRR** and **Precision@10**.


In [ ]:
from pathlib import Path
import random
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Works when launched from repo root or from inside recsys/.
if (Path.cwd() / "recsys").exists():
    RECSYS_DIR = Path.cwd() / "recsys"
else:
    RECSYS_DIR = Path.cwd()

DATA_DIR = RECSYS_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / "ml-100k.zip"
EXTRACT_DIR = DATA_DIR / "ml-100k"
RATINGS_PATH = EXTRACT_DIR / "u.data"

if not RATINGS_PATH.exists():
    if not ZIP_PATH.exists():
        url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
        print(f"Downloading {url} to {ZIP_PATH}...")
        urllib.request.urlretrieve(url, ZIP_PATH)
    print(f"Extracting {ZIP_PATH} to {DATA_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

print(f"Ratings file: {RATINGS_PATH}")


In [ ]:
ratings = pd.read_csv(
    RATINGS_PATH,
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
    engine="python",
)
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")

POSITIVE_THRESHOLD = 4
interactions = (
    ratings[ratings["rating"] >= POSITIVE_THRESHOLD]
    .sort_values("timestamp")
    .drop_duplicates(["user_id", "item_id"], keep="last")
    .reset_index(drop=True)
)

print(f"All rows: {len(ratings):,}")
print(f"Positive implicit interactions (rating >= {POSITIVE_THRESHOLD}): {len(interactions):,}")
print(
    f"Users: {interactions['user_id'].nunique()}, Items: {interactions['item_id'].nunique()},"
    f"Date range: {interactions['datetime'].min()} -> {interactions['datetime'].max()}"
)
interactions.head()


In [ ]:
def temporal_user_holdout(df, eval_ratio=0.1):
    train_parts = []
    eval_parts = []

    for _, group in df.sort_values("timestamp").groupby("user_id"):
        n = len(group)
        if n < 2:
            train_parts.append(group)
            continue

        n_eval = max(1, int(np.ceil(n * eval_ratio)))
        n_eval = min(n_eval, n - 1)

        train_parts.append(group.iloc[:-n_eval])
        eval_parts.append(group.iloc[-n_eval:])

    train_df = pd.concat(train_parts).sort_values("timestamp").reset_index(drop=True)
    eval_df = pd.concat(eval_parts).sort_values("timestamp").reset_index(drop=True)
    return train_df, eval_df

train_df, eval_df = temporal_user_holdout(interactions, eval_ratio=0.1)

user_ids = np.sort(train_df["user_id"].unique())
item_ids = np.sort(train_df["item_id"].unique())
user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}

train_df = train_df.assign(
    u=train_df["user_id"].map(user_to_idx),
    i=train_df["item_id"].map(item_to_idx),
)

eval_df = eval_df[
    eval_df["user_id"].isin(user_to_idx) & eval_df["item_id"].isin(item_to_idx)
].copy()
eval_df = eval_df.assign(
    u=eval_df["user_id"].map(user_to_idx),
    i=eval_df["item_id"].map(item_to_idx),
)

num_users = len(user_ids)
num_items = len(item_ids)

train_matrix = csr_matrix(
    (
        np.ones(len(train_df), dtype=np.float32),
        (train_df["u"].to_numpy(), train_df["i"].to_numpy()),
    ),
    shape=(num_users, num_items),
)

train_user_items = train_df.groupby("u")["i"].apply(lambda s: set(s.astype(int).tolist())).to_dict()
eval_user_items = eval_df.groupby("u")["i"].apply(lambda s: set(s.astype(int).tolist())).to_dict()
eval_users = sorted(u for u in eval_user_items if eval_user_items[u] and train_user_items.get(u))

print(f"Train interactions: {len(train_df):,}")
print(f"Eval interactions: {len(eval_df):,}")
print(f"Users in eval: {len(eval_users):,}")
print(f"Matrix shape: {train_matrix.shape}")


In [ ]:
def evaluate_model(score_fn, eval_users, eval_user_items, train_user_items, num_items, k=10):
    precision_scores = []
    rr_scores = []

    for u in eval_users:
        positives = eval_user_items.get(u, set())
        if not positives:
            continue

        seen = train_user_items.get(u, set())
        scores = score_fn(u).astype(np.float64, copy=True)
        if seen:
            scores[list(seen)] = -np.inf

        ranked_items = np.argsort(-scores)
        top_k = ranked_items[:k]

        precision_scores.append(len(set(top_k) & positives) / float(k))

        rank_positions = np.empty(num_items, dtype=np.int32)
        rank_positions[ranked_items] = np.arange(1, num_items + 1, dtype=np.int32)
        best_rank = min(rank_positions[list(positives)])
        rr_scores.append(1.0 / float(best_rank))

    return {
        "MRR": float(np.mean(rr_scores)),
        f"Precision@{k}": float(np.mean(precision_scores)),
        "n_eval_users": int(len(rr_scores)),
    }


## 1) Item-item collaborative filtering


In [ ]:
item_similarity = cosine_similarity(train_matrix.T, dense_output=True)
np.fill_diagonal(item_similarity, 0.0)
itemcf_scores = np.asarray(train_matrix @ item_similarity, dtype=np.float32)

itemcf_metrics = evaluate_model(
    score_fn=lambda u: itemcf_scores[u],
    eval_users=eval_users,
    eval_user_items=eval_user_items,
    train_user_items=train_user_items,
    num_items=num_items,
    k=10,
)
itemcf_metrics


## 2) Matrix factorization (truncated SVD)


In [ ]:
latent_dim = min(32, min(train_matrix.shape) - 1)
user_factors, singular_vals, item_factors_t = svds(train_matrix.astype(np.float32), k=latent_dim)

order = np.argsort(-singular_vals)
user_factors = user_factors[:, order]
singular_vals = singular_vals[order]
item_factors_t = item_factors_t[order, :]

mf_scores = np.asarray((user_factors * singular_vals) @ item_factors_t, dtype=np.float32)

mf_metrics = evaluate_model(
    score_fn=lambda u: mf_scores[u],
    eval_users=eval_users,
    eval_user_items=eval_user_items,
    train_user_items=train_user_items,
    num_items=num_items,
    k=10,
)
mf_metrics


## 3) Deep learning pointwise ranker


In [ ]:
def build_pointwise_dataset(train_user_items, num_items, negatives_per_positive=4, seed=SEED):
    rng = np.random.default_rng(seed)
    users = []
    items = []
    labels = []

    for u, positives in train_user_items.items():
        if not positives:
            continue

        for i in positives:
            users.append(u)
            items.append(i)
            labels.append(1.0)

            neg_added = 0
            while neg_added < negatives_per_positive:
                j = int(rng.integers(0, num_items))
                if j in positives:
                    continue
                users.append(u)
                items.append(j)
                labels.append(0.0)
                neg_added += 1

    return (
        np.array(users, dtype=np.int64),
        np.array(items, dtype=np.int64),
        np.array(labels, dtype=np.float32),
    )


class PointwiseMLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=48, hidden_dims=(128, 64), dropout=0.15):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        layers = []
        in_dim = emb_dim * 2
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_ids, item_ids):
        user_vec = self.user_emb(user_ids)
        item_vec = self.item_emb(item_ids)
        x = torch.cat([user_vec, item_vec], dim=-1)
        return self.mlp(x).squeeze(-1)


train_users_arr, train_items_arr, train_labels_arr = build_pointwise_dataset(
    train_user_items=train_user_items,
    num_items=num_items,
    negatives_per_positive=4,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, pointwise examples: {len(train_labels_arr):,}")

train_dataset = TensorDataset(
    torch.from_numpy(train_users_arr),
    torch.from_numpy(train_items_arr),
    torch.from_numpy(train_labels_arr),
)
train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)

pointwise_model = PointwiseMLP(num_users=num_users, num_items=num_items).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(pointwise_model.parameters(), lr=1e-3)

num_epochs = 5
for epoch in range(num_epochs):
    pointwise_model.train()
    total_loss = 0.0
    total_examples = 0

    for batch_users, batch_items, batch_labels in train_loader:
        batch_users = batch_users.to(device)
        batch_items = batch_items.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        logits = pointwise_model(batch_users, batch_items)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_users.size(0)
        total_examples += batch_users.size(0)

    print(f"Epoch {epoch + 1}/{num_epochs} | loss={total_loss / total_examples:.4f}")


In [ ]:
all_item_tensor = torch.arange(num_items, dtype=torch.long, device=device)

@torch.no_grad()
def deep_scores_for_user(u):
    pointwise_model.eval()
    user_tensor = torch.full((num_items,), u, dtype=torch.long, device=device)
    logits = pointwise_model(user_tensor, all_item_tensor)
    return logits.detach().cpu().numpy()

deep_metrics = evaluate_model(
    score_fn=deep_scores_for_user,
    eval_users=eval_users,
    eval_user_items=eval_user_items,
    train_user_items=train_user_items,
    num_items=num_items,
    k=10,
)
deep_metrics


In [ ]:
results = pd.DataFrame([
    {"Model": "Item-item CF", **itemcf_metrics},
    {"Model": "Matrix factorization (SVD)", **mf_metrics},
    {"Model": "Deep pointwise ranker", **deep_metrics},
]).set_index("Model")

results = results[["MRR", "Precision@10", "n_eval_users"]].sort_values("MRR", ascending=False)
results


In [ ]:
ax = results[["MRR", "Precision@10"]].plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_title("Offline ranking metrics on MovieLens 100K")
ax.set_ylabel("Metric value")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
